In [1]:
from dotenv import load_dotenv

load_dotenv()

True

## Sort your agent facts

Use the same split here as in the runtime-context lesson:

- Context = stable extras that describe the run.
- State = facts the agent should learn and remember across turns.

The field `favourite_colour` is a perfect example of state: it changes when the user tells us something new.

In [ ]:
from langchain.agents import AgentState

class CustomState(AgentState):
    favourite_colour: str = "unknown"

## Write to state

The agent will update the state only when a tool returns a `Command(update=...)` payload.

In [ ]:
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage

@tool
def update_favourite_colour(favourite_colour: str, runtime: ToolRuntime) -> Command:
    """Update the user's favourite colour in the agent state after they reveal it."""
    return Command(
        update={
            "favourite_colour": favourite_colour,
            "messages": [
                ToolMessage(
                    "Successfully updated favourite colour",
                    tool_call_id=runtime.tool_call_id,
                )
            ],
        }
    )

In [12]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent_with_state = create_agent(
    "groq:openai/gpt-oss-20b",
    tools=[update_favourite_colour],
    checkpointer=InMemorySaver(),
    state_schema=CustomState,
)

In [13]:
from langchain.messages import HumanMessage

response = agent_with_state.invoke(
    {"messages": [HumanMessage(content="My favourite colour is green")]},
    {"configurable": {"thread_id": "state-1"}},
)

print("Stored state:", response.get("favourite_colour"))
print(response["messages"][-1].content)

Stored state: green
Got it—green is your favourite colour! Anything else you'd like to share or ask about?


In [6]:
from pprint import pprint

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='My favourite colour is green', additional_kwargs={}, response_metadata={}, id='cacb9a90-dcd8-4390-a0b0-8395d9c71833'),
              AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to call the function update_favourite_colour with favourite_colour: "green".', 'tool_calls': [{'id': 'fc_549c1fc4-b927-4f3e-9df3-42750c6d9d8b', 'function': {'arguments': '{"favourite_colour":"green"}', 'name': 'update_favourite_colour'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 45, 'prompt_tokens': 137, 'total_tokens': 182, 'completion_time': 0.050407131, 'completion_tokens_details': {'reasoning_tokens': 18}, 'prompt_time': 0.007920434, 'prompt_tokens_details': None, 'queue_time': 0.147960306, 'total_time': 0.058327565}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_228717f27c', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provi

In [14]:
response = agent_with_state.invoke(
    {
        "messages": [HumanMessage(content="Hello, how are you?")],
        "favourite_colour": "green",
    },
    {"configurable": {"thread_id": "state-10"}},
)

print(response.get("favourite_colour"))
print(response["messages"][-1].content)

green
Hello! I’m doing great, thanks for asking. How can I help you today?


## Read state

Now add a read tool, then ask the agent for the same fact again. The thread state should be reused, so it should recall what was learned earlier.

In [15]:
@tool
def read_favourite_colour(runtime: ToolRuntime) -> str:
    """Read the user's favourite colour from the agent state."""
    try:
        return runtime.state["favourite_colour"]
    except KeyError:
        return "No favourite colour found in state"

agent_with_state_and_reader = create_agent(
    "groq:openai/gpt-oss-20b",
    tools=[update_favourite_colour, read_favourite_colour],
    checkpointer=InMemorySaver(),
    state_schema=CustomState,
)

In [16]:
response = agent_with_state_and_reader.invoke(
    {"messages": [HumanMessage(content="My favourite colour is green")]},
    {"configurable": {"thread_id": "state-1"}},
)

print("State after learning:", response.get("favourite_colour"))
print(response["messages"][-1].content)

State after learning: green
Got it—green is your favorite colour! 🌱 Is there anything special about green that you love, or any projects or outfits where you love to use it?


In [17]:
response = agent_with_state_and_reader.invoke(
    {"messages": [HumanMessage(content="What's my favourite colour?")]},
    {"configurable": {"thread_id": "state-1"}},
)

print("State recall:", response.get("favourite_colour"))
print(response["messages"][-1].content)

State recall: green
Your favourite colour is green. 🌿
